In [1]:
# from IPython.core.interactiveshell import InteractiveShell
# InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import seaborn as sns
sns.set()
import gc
gc.collect()
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import set_matplotlib_formats
from IPython.display import clear_output
set_matplotlib_formats('retina')
#from tqdm import tqdm
#tqdm.pandas()
import numpy as np
from pyhive import presto
from datetime import datetime, timedelta
# from bson import ObjectId
from functools import reduce
from sklearn.cluster import KMeans
# from pymongo import MongoClient
import glob
import warnings
warnings.filterwarnings('ignore')
from datetime import date
import json
import re
#import dtale
#from h3 import h3
# import pandasql as ps

In [2]:
metabase_connection = presto.connect(
        # host='bi-trino-2.serving.data.production.internal',
#         host='bi-trino.serving.data.production.internal',
        #host='bi-presto.serving.data.production.internal',
        #host='prime-trino.serving.data.production.internal',
         host="processing-2.processing.data.production.internal",
#         host="presto.processing.yoda.run",
        #host="processing.processing.data.production.internal",

        port=80,
        protocol='http',
        catalog ='hive',
        username='abhishek.akki@rapido.bike')

In [8]:

query = """


with exp_base as (

select 
-- *
captain_id as captain_id_exp,
city,
mobileNumber,
performance_segment as performance_segment_exp,
consistency_segment as consistency_segment_exp,
"Group" as exp_group

from hive.experiments_internal.captain_byob_exp_bases_six_cities_v1 
where city in ('Ahmedabad','Chandigarh')
and "Group" = 'Test'
-- where taxi_order_earnings > 0
-- limit 100


),

selectors as (
select 
-- * 
distinct name, userid, yyyymmdd
-- selectorid, name 
-- userid, active, type, usertype, yyyymmdd, selectorid, name 
from canonical.user_selector
where 
-- createdat_utc = '20250120'
 yyyymmdd >= '20250110'
and type = 'custom'
and usertype = 'rider'
and active = true
and name
-- like ('%AHMD%')
-- city in ('Vijayawada','Hyderabad','Ahmedabad','Chandigarh','Kolkata','Vishakapatnam')

in (
'AHMD S W04 7DA LP',
'AHMD S W04 7DA MP',
'AHMD S W04 7DA HP',
'AHMD S W04 714DIA',
'AHMD S W04 1430DIA',
'AHMD S W04 30DIA',
'AHMD S W04 ZT',
'CHD S W04 14DA LP',
'CHD S W04 14DA MP',
'CHD S W04 14DA HP',
'CHD S W04 14DA UHP',
'CHD S W04 1445DIA LP',
'CHD S W04 1445DIA MP',
'CHD S W04 1445DIA HP',
'CHD S W04 1445DIA UHP',
'CHD S W04 45DIA'
)
-- limit 10
) 


select captain_id_exp, city, mobileNumber, name, userid, yyyymmdd 
from exp_base a 
left join selectors b 
on a.captain_id_exp = b.userid



"""

df = pd.read_sql(query, metabase_connection)


In [9]:
df.head(20)

,captain_id_exp,city,mobileNumber,name,userid,yyyymmdd
0,5bd19338a098e60a3ea82489,Ahmedabad,9712093961,AHMD S W04 714DIA,5bd19338a098e60a3ea82489,20250118
1,62bc070b7d7967d755f690d5,Ahmedabad,9166632843,AHMD S W04 7DA LP,62bc070b7d7967d755f690d5,20250118
2,64c7953d56cd998dff04f280,Ahmedabad,8780246239,AHMD S W04 714DIA,64c7953d56cd998dff04f280,20250118
3,63a936d4413050c8c77656c3,Ahmedabad,9024310167,AHMD S W04 714DIA,63a936d4413050c8c77656c3,20250118
4,63ce764c9a65bc784882ef14,Ahmedabad,7046382423,AHMD S W04 714DIA,63ce764c9a65bc784882ef14,20250118
5,6662f17ff883d56749b8339d,Ahmedabad,9974046663,AHMD S W04 7DA LP,6662f17ff883d56749b8339d,20250118
6,5f469af17f36b6d26f2aff25,Ahmedabad,7405259572,AHMD S W04 7DA LP,5f469af17f36b6d26f2aff25,20250118
7,5d908c26a1f0a71c604ea2f2,Ahmedabad,8680991961,AHMD S W04 7DA MP,5d908c26a1f0a71c604ea2f2,20250118
8,618d7843c98cb39d54fc1d44,Ahmedabad,7990677352,AHMD S W04 7DA LP,618d7843c98cb39d54fc1d44,20250118
9,674eb142f56fde720ab5e25a,Ahmedabad,9356527150,AHMD S W04 714DIA,674eb142f56fde720ab5e25a,20250118


In [10]:
df['userid'].nunique()

1915

In [11]:
df.to_csv('ahm_chd.csv', index = False)